In [1]:

# Demo: insert_kmers replaces a zero-length marker <bc/> with <bc>[kmer]</bc>
import poolparty as pp
pp.init()
template_pool = (
    pp.from_seq('TCCCGACT<cre>GGAAAGCGGGC<misc/>AGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA')
    .named('template_pool')
)

wt_pool = (
    template_pool
    .repeat_states(3, seq_name_prefix='wt.v')
    .named('wt_pool')
)

del_pool = (
    template_pool
    .deletion_scan('cre', deletion_length=5, positions=slice(None, None, 5), mode='hybrid', num_hybrid_states=3, seq_name_prefix='del_')
    .repeat_states(3, seq_name_prefix='v')
    .named('del_pool')
)

shuf_pool = (
    template_pool
    .shuffle_seq('cre', mode='hybrid', num_hybrid_states=5, seq_name_prefix='shuff_')
    .repeat_states(3, seq_name_prefix='v')
    .named('shuf_pool')
)

mut_pool = (
    template_pool
    .mutagenize('cre', mutation_rate=0.1, mode='hybrid', num_hybrid_states=5, seq_name_prefix='mut_')
    .repeat_states(3, seq_name_prefix='v')
    .named('mut_rate_pool')
)

site_pool = pp.from_seqs(['AAAAA', 'CCCCC'], mode='sequential', seq_name_prefix='site_')

repl_pool = (
    template_pool
    .replacement_scan('cre', ins_pool=site_pool, positions=slice(None, None, 5), mode='sequential', seq_name_prefix='repl_')
    .repeat_states(3, seq_name_prefix='v')
    .named('repl_pool')
)

combo_pool = (
    pp.stack([
        wt_pool, 
        shuf_pool,
        mut_pool,
        del_pool,
        repl_pool,
    ])
    .insert_kmers('bc', length=5)
    .named('combo_pool')
    .clear_markers()
    .upper()
    .print_library()
)

pool[17]: seq_length=None, num_values=102
state  name              seq
    0  wt.v0             TCCCGACTGGAAAGCGGGCAGTGAGCACACAGGAAAATTACGGTGGCCAGATCGGA
    1  wt.v1             TCCCGACTGGAAAGCGGGCAGTGAGCACACAGGAAAATTACGGAAAATAGATCGGA
    2  wt.v2             TCCCGACTGGAAAGCGGGCAGTGAGCACACAGGAAAATTACGGGTGGTAGATCGGA
    3  shuff_0.v0        TCCCGACTGCCTGAGGCGCGAAAGGGCAAGAAAAGAATTACGGGGGGTAGATCGGA
    4  shuff_0.v1        TCCCGACTGCCTGAGGCGCGAAAGGGCAAGAAAAGAATTACGGCTGACAGATCGGA
    5  shuff_0.v2        TCCCGACTGCCTGAGGCGCGAAAGGGCAAGAAAAGAATTACGGTGATGAGATCGGA
    6  shuff_1.v0        TCCCGACTCGAAGAAGGGGAGGGTAACAAGAACCCGATTACGGTAATAAGATCGGA
    7  shuff_1.v1        TCCCGACTCGAAGAAGGGGAGGGTAACAAGAACCCGATTACGGGACCCAGATCGGA
    8  shuff_1.v2        TCCCGACTCGAAGAAGGGGAGGGTAACAAGAACCCGATTACGGCAAAAAGATCGGA
    9  shuff_2.v0        TCCCGACTGGCAGAGAAGACCGGGACGCGATAGAAAATTACGGGGGCGAGATCGGA
   10  shuff_2.v1        TCCCGACTGGCAGAGAAGACCGGGACGCGATAGAAAATTACGGTCCTTAGATCGGA
   11  shuff_2.v2        TC

In [3]:
combo_pool.print_dag(show_pools=False)

op[17]:upper [mode=fixed, n=1]
└── op[16]:fixed [mode=fixed, n=1]
    └── combo_pool.op [mode=random, n=1]
        └── op[14]:stack [mode=fixed, n=1]
            ├── wt_pool.op [mode=sequential, n=3]
            │   └── template_pool.op [mode=fixed, n=1]
            ├── shuf_pool.op [mode=sequential, n=3]
            │   └── op[5]:shuffle_seq [mode=hybrid, n=5]
            │       └── template_pool.op [mode=fixed, n=1]
            ├── mut_rate_pool.op [mode=sequential, n=3]
            │   └── op[7]:mutagenize [mode=hybrid, n=5]
            │       └── template_pool.op [mode=fixed, n=1]
            ├── del_pool.op [mode=sequential, n=3]
            │   └── op[3]:deletion_scan(from_seq) [mode=fixed, n=1]
            │       └── op[2]:deletion_scan(marker_scan) [mode=hybrid, n=3]
            │           └── template_pool.op [mode=fixed, n=1]
            └── repl_pool.op [mode=sequential, n=3]
                └── op[12]:replacement_scan(replace_marker_content) [mode=fixed, n=1]
          

Pool(id=17, name='pool[17]', op='op[17]:upper', num_values=102)

In [4]:
combo_pool.counter.print_dag()

pool[17].state (counter, io=0, n=102)
└── [op=Product]
    ├── op[17]:upper.state (counter, io=0, n=1)
    ├── op[16]:fixed.state (counter, io=0, n=1)
    ├── combo_pool.op.state (counter, io=0, n=1)
    └── pool[14].state (counter, io=0, n=102)
        └── [op=Stack]
            ├── wt_pool.state (counter, io=0, n=3)
            │   └── [op=Product]
            │       ├── wt_pool.op.state (counter, io=0, n=3)
            │       └── template_pool.state (counter, io=0, n=1)
            │           └── [op=Passthrough]
            │               └── template_pool.op.state (counter, io=0, n=1)
            ├── shuf_pool.state (counter, io=0, n=15)
            │   └── [op=Product]
            │       ├── shuf_pool.op.state (counter, io=0, n=3)
            │       ├── op[5]:shuffle_seq.state (counter, io=0, n=5)
            │       └── template_pool.state (counter, io=0, n=1)
            │           └── [op=Passthrough]
            │               └── template_pool.op.state (counter, io=0